# 1. Loading Essential Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2. Loading the Dataset 

In [ ]:
#df_csv = pd.read_csv('data.csv') # Reading from csv file
df = pd.read_excel('b20_data.xlsx') # Reading from excel file
#df_json = pd.read_json('data.json') # Reading from json file

# 3. Data Inspection

In [ ]:
# Quick Overview

def quick_overview(df):
    print('=' * 50)
    print('      Data Information and Summary')
    print('=' * 50)

    info = df.info() # Get info about data types and non-null counts
    print(info)
    
    print('')
    print('=' * 50)
    print('      Data Summary and Structure')
    print('=' * 50)

    des = df.describe() # Get statistical summary of numerical columns
    print(des)


    print('=' * 50)
    print('      Data Shape and Columns')
    print('=' * 50)

    print(f'Data shape: {df.shape}')
    print(f'Column names: {list(df.columns)}')

    print('')
    print('=' * 50)
    print('      Viewing Data Samples')
    print('=' * 50)

    print('First 5 rows:')
    print(df.head()) # View first few rows of the dataset
    print('\nLast 5 rows:')
    print(df.tail()) # View last few rows of the dataset
    
    return

quick_overview(data)

In [ ]:
# Identify Problems
null_counts = df.isnull().sum() # Check for missing values
dup = df.duplicated().sum() # Check for duplicate rows
vc = df['pressure'].value_counts() # Check for inconsistent categorical values
out = df['pressure'].plot(kind='box') # Check for outliers in numerical data

print('=' * 50)
print('      Missing Values Report')
print('=' * 50)
print(f'Missing values per column:\n{null_counts}')

print('')
print('=' * 50)
print('      Duplicate Rows Report')
print('=' * 50)
print(f'Duplicate rows: {dup}')

print('')
print('=' * 50)
print('      Inconsistent Categorical Values Report')
print('=' * 50)
print(f'Inconsistent categorical values in column "pressure":\n{vc}')
print('')

# 4. Remove Duplicate Records

In [ ]:
df = df.drop_duplicates() # Remove all duplicate rows
print(f'Rows after removing duplicates: {df.shape[0]}')

In [ ]:
df = df.drop_duplicates(subset=['pressure'], keep='first') # Remove duplicate based on specified column
print(f'Rows after removing duplicates based on "pressure": {df.shape[0]}')

# 5. Handling Missing Data

In [ ]:
# Step1: Fill withmeaningful defaults
df['pressure'] = df['pressure'].fillna(0)
df['notes'] = df['notes'].fillna("No comment")

In [ ]:
# Step 2: Drop rows with critical missing values
df = df.dropna(subset=['pressure']) # Can add multiple columns if needed
print(f'Rows after dropping rows with missing "pressure": {df.shape[0]}')

In [ ]:
# step 3: Fill missing values with mean/median/mode
# For numerical columns, we can use mean or median
df['pressure'] = df['pressure'].fillna(df['pressure'].mean()) # Using mean to fill missing values

In [ ]:
# Step 4: For categorical columns, we can use mode
df['notes'] = df['notes'].fillna(df['notes'].mode()[0])

In [ ]:
# Step 5:Forward/backward fill for time series
df = df.fillna(method='ffill') # Forward fill
df = df.fillna(method='bfill') # Backward fill

# 6. Standardize Text Data

In [ ]:
# fix case inconsistencies in categorical data
df['pressure'] = df['pressure'].str.lower() # Convert to lowercase
df['pressure'] = df['pressure'].str.upper() # Convert to uppercase
df['pressure'] = df['pressure'].str.title() # Convert to title case

In [ ]:
# Remove extra white spaces
df['pressure'] = df['pressure'].str.strip() # Remove leading and trailing white spaces
df['notes'] = df['notes'].str.replace(r'\s+', '', regex=True) # Remove leading and trailing white spaces

In [ ]:
# Standardize with replace
df['pressure'] = df['pressure'].replace({'low': 'Low', 'medium': 'Medium', 'high': 'High'}) # Replace inconsistent values with standardized ones

# 7. Fix Data Types

In [ ]:
# Convert to correct data types
df['pressure'] = pd.to_numeric(df['pressure'], errors='coerce') # Convert to numeric, coercing errors to NaN
df['pressure'] = df['pressure'].astype(float) # Convert to float
df['pressure'] = df['pressure'].astype(int) # Convert to integer
df['pressure'] = df['pressure'].astype(str) # Convert to string
df['pressure'] = df['pressure'].astype('bool') # Convert to boolean

In [ ]:
# Parse dates properly
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce') # Convert to datetime, coercing errors to NaT
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d', errors='coerce') # Convert to datetime with specific format

In [ ]:
# Extract date components
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day_name() # Get day name instead of day number
df['order_weekday'] = df['order_date'].dt.weekday_name() # Get weekday name instead of weekday number


# 8. Remove Invalid Data

In [ ]:
df = df[df['age'] > 0] # Remove rows with invalid age values (negative or zero)

In [ ]:
# Drop columns with too many missing values or irrelevant information
threshold = 0.5 # Set threshold for dropping columns (e.g., drop if more than 50% missing)
#df = df.dropna(thresh=int(threshold * len(df)), axis=1) # Drop columns that have more than 50% missing values   

missing_pct = df.isnull().mean() * 100 # Calculate percentage of missing values per column
print(f'Percentage of missing values per column:\n{missing_pct}')
col_to_drop = missing_pct[missing_pct > 50].index # Get columns that have more than 50% missing values
print(f'Columns to drop due to high missing values:\n{col_to_drop}')
df = df.drop(columns=col_to_drop) # Drop columns with too many missing values

# 9. Handle Outliers

In [ ]:
# Detect outliers using IQR method
Q1 = df['pressure'].quantile(0.25)
Q3 = df['pressure'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df = df[(df['pressure'] >= lower_bound) & (df['pressure'] <= upper_bound)]

In [ ]:
# Remove outliers 
df = df[(df['pressure'] >= lower_bound) & (df['pressure'] <= upper_bound)] # Keep rows where pressure is within IQR bounds
df

In [ ]:
# Remove outliers using Z-score method
from scipy import stats 
z_scores = stats.zscore(df['pressure'])
abs_z_scores = np.abs(z_scores)
lower_bound = 3 # Set Z-score threshold for outlier detection
df = df[abs_z_scores < lower_bound] # Keep rows where Z-score is less than threshold, effectively removing outliers
df

In [ ]:
# Cap Outliers (winsorization)

df['pressure'] = df['pressure'].clip(lower_bound, upper_bound) # Cap values outside the bounds to the respective bound
df

# 10. Renaming and Reorganizing columns

In [ ]:
# Renaming specific columns for clarity
df = df.rename(columns={'pressure': 'blood_pressure', 'notes': 'patient_notes'})

In [ ]:
# Clean all column names to be lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')
df

# 11. Validate and Export

In [ ]:
# Final check for validity
print('=' * 50)
print('      Final Data Overview After Cleaning')
print('=' * 50)

print(f' Missing Values:{df.isnull().sum()}')
print(f'Duplicates:{df.duplicated().sum()}')
print(f'Memory usage:{df.memory_usage(deep=True)}')

      Final Data Overview After Cleaning
 Missing Values:datetime    0
pressure    0
dtype: int64
Duplicates:10505
Memory usage:Index          132
datetime    204336
pressure    204336
dtype: int64


In [ ]:
# Export cleaned data to new file
df.to_csv('cleaned_data.csv', index=False) # Export to CSV without index
df.to_excel('cleaned_data.xlsx', index=False) # Export to Excel without index
